In [1]:
from IRCoT import graph_builder

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../dfs/final_df.csv")
df.sample(3)

,question,evidence,ground_truth,url,source
13,Why do the proposed arborescence-based cross-e...,"[""We also note that, unlike on MedMentions, no...",The proposed arborescence models rely on expli...,https://aclanthology.org/2022.naacl-main.343.pdf,1
6,What is the source of the dataset?,['Prior research [ 93 ] shows that Youtube vid...,The dataset was collected on Youtube.,https://openreview.net/forum?id=eJhc_CPXQIT,3
17,What BLEU score did the baseline model develop...,"[""Table 2: The Transformer achieves better BLE...","The Transformer base model achieved 27.3 BLEU,...",attention is all you need & RoPE,2


In [3]:
from datetime import datetime
import json
from pathlib import Path

db_dir = {
    "1": "../outputs/test1/",
    "2": "../outputs/test2/",
    "3": "../outputs/test3/"
}

In [4]:
start = datetime.now()
all_answers = []
qa_latencies = []

for db_id, input_dir in db_dir.items():
    print(f"Running pipeline for DB {db_id}...")
    rows = df[df["source"] == int(db_id)]

    for _, row in rows.iterrows():
        graph = graph_builder()
        question = row["question"]
        print(f"Question: {question}")
        s = datetime.now()
        result = graph.invoke({
            "query": question,
            "iteration": 0,
            "max_iterations": 5,
            "retrieved_docs": [],
            "sub_queries": [],
            "is_sufficient": False,
            "reason_for_judgment": [],
            "db_path": input_dir,
            "top_k": 3,
        })

        log = {
        "timestamp": datetime.now().isoformat(),
        "query": result["query"],
        "answer": result["answer"],
        "final_answer": result["answer"][-1],
        "iterations": result["iteration"],
        "sub_queries": result["sub_queries"],
        "reason_for_judgment": result["reason_for_judgment"],
        "retrieved_docs": result["retrieved_docs"],
        "all_answers": result["answer"],
        }

        log_path = f"logs/langraph/rag_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        Path(log_path).parent.mkdir(parents=True, exist_ok=True)
        with open(log_path, "w") as f:
            json.dump(log, f, indent=2)

        print(f"Time taken for question: {(datetime.now() - s).seconds} seconds")
        qa_latencies.append((datetime.now() - s).seconds)
        unique_docs = set(tuple(x) if isinstance(x, list) else x for x in result.get("retrieved_docs", []))
        all_answers.append({
                "question": question,
                "answer": result.get("answer", ""),
                "context": unique_docs,
                "ground_truth": row["ground_truth"],
                "evidence": row["evidence"],
                "source": db_id
            })
end = datetime.now()
ellapsed_time = end - start
print(f"Total time taken: {ellapsed_time}")

Running pipeline for DB 1...
Question: What is the target KB size used for MedMentions in the experiments?
Judge response: {'is_sufficient': True, 'reason': 'Directly states the target KB size used: 2.3 million entities (2.3M) in the MedMentions KB.'}
Time taken for question: 11 seconds
Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Judge response: {'is_sufficient': False, 'reason': 'The answer does not determine whether training and inference batches contain mentions from the same document. It only states that the information is not specified, offering no direct conclusion or supporting details about batch composition.'}
Judge response: {'is_sufficient': True, 'reason': 'The answer directly states that the available context does not specify batch composition, thus clarifying that we cannot determine whether training and inference batches include mentions from the same document.'}
Time taken f

In [5]:
max(qa_latencies), min(qa_latencies), sum(qa_latencies)/len(qa_latencies)

(71, 7, 22.736842105263158)

In [6]:
qa_latencies

[11, 37, 18, 16, 15, 17, 23, 16, 27, 13, 16, 23, 12, 12, 18, 9, 71, 7, 71]

In [9]:
all_answers_df = pd.DataFrame(all_answers)
all_answers_df.to_csv("../dfs/all_answers_iterative.csv", index=False)

In [22]:
df_with_eval = all_answers_df.copy()

In [12]:
def normalize_text(value):
    # `answer` can be a list of iterative answers; keep the latest non-empty one.
    if isinstance(value, list):
        for item in reversed(value):
            if isinstance(item, str) and item.strip():
                return item
        return " ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)

def f1_token_overlap(pred, truth):
    pred_text = normalize_text(pred)
    truth_text = normalize_text(truth)
    pred_tokens = set(pred_text.lower().split())
    truth_tokens = set(truth_text.lower().split())
    if not pred_tokens or not truth_tokens:
        return 0
    return 2 * len(pred_tokens & truth_tokens) / (len(pred_tokens) + len(truth_tokens))

all_answers_df['f1'] = all_answers_df.apply(
    lambda row: f1_token_overlap(row['answer'], row['ground_truth']), axis=1
)

all_answers_df.sample(5)

,question,answer,context,ground_truth,evidence,source,f1
6,During the constrained clustering procedure fo...,[Cluster correctness and dissimilarity rules: ...,{(Doc: \[\nE _ {m _ {i}} = \left\{\left(e _ {i...,The system partitions the graph into disjoint ...,"[""Forming Clusters for Positive Sampling The g...",1,0.394495
2,"Does the model need to compute \psi(m_i, e) fo...","[No. psi(e, m_i) is defined as Enc_E(e)^T Enc_...",{(Doc: and the weight of the edge from entity ...,No.,"['For each mentionm , construct Em by ( a ) ad...",1,0.044444
18,Is the dataset under a Creative Commons license?,[The retrieved context does not provide any li...,"{(Doc: [1] Jia Deng, Wei Dong, Richard Socher,...",Yes.,"['Specifically , we will use Attribution 4.0 I...",3,0.000000
16,What ethical considerations were taken into ac...,[The retrieved context does not mention or spe...,{(Doc: new annotation schema that can be easil...,"Firstly, we carefully selected gender-neutral,...",['The key abstraction of MOMA-LRG is the activ...,3,0.448598
1,Do the training and inference batches contain ...,[The retrieved context does not specify whethe...,{(Doc: ### A Appendix\n\n#### A.1 Experiment D...,"Yes, the batches may contain mentions from the...","['We optimize f ( · , · ) using mini-batch gra...",1,0.241379


In [21]:
all_answers_df['answer'].iloc[5][-1]

'The gains come from explicitly modeling coreference relations during training. ZeShEL has much smaller coreference clusters than MedMentions (in ZeShEL, clusters are typically at most 3 mentions, whereas MedMentions has clusters with up to 1256, 434, and 447 mentions). This statistical difference in cluster size—the number of coreference links per entity—means there are far fewer coreference links for the arborescence-based models to exploit in ZeShEL, leading to mixed results compared to mention-entity baselines, while MedMentions provides many more links and shows consistent gains.'

In [20]:
from rag_eval import evaluate

def normalize_answer(value):
    if isinstance(value, list):
        for item in reversed(value):
            if isinstance(item, str) and item.strip():
                return item
        return " ".join(str(item) for item in value)
    if pd.isna(value):
        return ""
    return str(value)

def normalize_contexts(value):
    if isinstance(value, set):
        return [str(item) for item in value]
    if isinstance(value, (list, tuple)):
        return [str(item) for item in value]
    if pd.isna(value):
        return []
    return [str(value)]

results = []
for row in all_answers_df.itertuples():
    print(f"Evaluating Question: {row.question}")
    result = evaluate(
        question=row.question,
        answer=normalize_answer(row.answer),
        contexts=normalize_contexts(row.context),
        ground_truth=row.ground_truth
    )
    results.append(result)

Evaluating Question: What is the target KB size used for MedMentions in the experiments?
Evaluating Question: Do the training and inference batches contain mentions from the same document to leverage coreferent mentions as much as possible?
Evaluating Question: Does the model need to compute \psi(m_i, e) for all entities in the target knowledge base during inference?
Evaluating Question: Why not use the complete graph for training instead of pruning it?
Evaluating Question: How does the arborescence-based training model calculate the dissimilarity between a mention and an entity, and what specific underlying models compute the edge weights for this calculation?
Evaluating Question: Why do the proposed arborescence-based cross-encoder models show consistent gains in linking accuracy on the MedMentions dataset but mixed results compared to mention-entity baselines on the ZeShEL dataset, and what specific statistical difference accounts for this?
Evaluating Question: During the constraine

In [23]:
for i, result in enumerate(results):
    for metric_name, metric_value in result.items():
        if isinstance(metric_value, dict):
            df_with_eval.at[i, metric_name] = metric_value.get("score")
        else:
            df_with_eval.at[i, metric_name] = metric_value

In [24]:
metrics = ["faithfulness", "context_relevance", "answer_relevance", "answer_correctness", "overall"]

for id in df_with_eval["source"].unique():
    curr = df_with_eval[df_with_eval["source"] == id]
    
    print(f"\n--- Mean Metrics for source {id} ---")
    print(curr[metrics].mean())


--- Mean Metrics for source 1 ---
faithfulness          0.958338
context_relevance     0.850000
answer_relevance      1.000000
answer_correctness    0.750000
overall               0.889587
dtype: float64

--- Mean Metrics for source 2 ---
faithfulness          1.000
context_relevance     0.900
answer_relevance      1.000
answer_correctness    1.000
overall               0.975
dtype: float64

--- Mean Metrics for source 3 ---
faithfulness          1.0000
context_relevance     0.7125
answer_relevance      1.0000
answer_correctness    0.4375
overall               0.7875
dtype: float64
